009 DATA CLEANING WEATHER CONDITIONS

- Start with 008_data_only_pedestrians parquet file
- End with 009_weather_included parquet file

OBJECTIVES: 
- clean 'Latitude' and 'Longitude' columns (N)
- clean 'CondizioneAtmosferica' column (O)
- integration of independent weather source


In [76]:
import pandas as pd
import numpy as np
import pyarrow
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

In [77]:
df = pd.read_parquet("008_data_only_pedestrians.parquet").copy()
df.shape

(19416, 46)

In [78]:
df.columns

Index(['Protocollo', 'DataOraIncidente', 'CondizioneAtmosferica', 'Gruppo',
       'Localizzazione1', 'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02',
       'Chilometrica', 'DaSpecificare', 'Latitude', 'Longitude',
       'parked_vehicle_involved', 'driver_injury', 'driver_gender',
       'passenger_no_of_females', 'passenger_no_of_males',
       'passenger_no_of_unknown_sex', 'passengers_killed',
       'passengers_uninjured', 'passengers_injured', 'phase_of_day', 'YEAR',
       'MONTH', 'DATE', 'TIME', 'DAY', 'pedestrian_gender', 'road_conditions',
       'road_markings_absent', 'road_markings_onroad',
       'road_markings_signposts', 'road_markings_temporary', 'road_features',
       'road_markings_traffic_lights', 'road_surface', 'accident_type',
       'TipoStradaDifficulty', 'visibility', 'pedestrian_injury',
       'lighting_insufficient', 'traffic_density', 'vehicle_type',
       'vehicle_unknown', 'hit_and_run'],
      dtype='object')

### (N) 'Latitude' & 'Longitude' columns

In order to request weather information, we need a Latitude and Longitude. 

Let us first examine the values in these columns to ensure they really refer to positions within the province of Rome. 

In [79]:
df.isnull().sum()[df.isnull().sum() > 0]

Latitude        2150
Longitude       2152
vehicle_type     637
dtype: int64

There are a large number of missing values. 

In [80]:
round(df[['Latitude', 'Longitude']].describe(), 5)

,Latitude,Longitude
count,17266.00000,17264.00000
mean,41.89069,12.49117
std,0.15357,0.16737
min,32.37145,9.64400
25%,41.86738,12.45140
50%,41.89459,12.48950
75%,41.91630,12.53610
max,45.39700,24.23083


The position of Rome, according to Google maps, is 41.8967° N, 12.4822° E, so it is apparent that the max and min values correspond to locations well away from Rome. Near Rome, a change of 1 in latitude corresponds to a distance of about 110km and a change of 1 in longitude corresponds to a ditance of about 84km.  To better illustrate variation in latitudes and longitudes, we can use boxplots:

In [81]:
def boxlatlong(df):
    """Draw boxplots for Latitude and Longitude with Rome center reference lines."""

    # Box plot for Latitude
    fig_lat = px.box(df, x="Latitude", width=800, height=200)
    fig_lat.add_vline(
        x=41.8967,
        line_color="red",
        line_dash="dash",
        line_width=2,
        annotation_text="Rome centre",
        annotation_position="top left"
    )
    fig_lat.show()

    # Box plot for Longitude
    fig_lon = px.box(df, x="Longitude", width=800, height=200)
    fig_lon.add_vline(
        x=12.4822,
        line_color="red",
        line_dash="dash",
        line_width=2,
        annotation_text="Rome centre",
        annotation_position="top left"
    )
    fig_lon.show()

In [82]:
boxlatlong(df)

Inverting the Longitude and Latitudes for the outliers still does not give us a position anywhere near Rome, so the longitude and latitude have not been inverted. 

Using a map, we define a generous area around Rome:

- 41.6 < LATITUDE < 42.1
- 12.1 < LONGITUDE < 12.8

We will now delete values outside of these boundaries, as they are clearly incorrect.

In [83]:
# Build mask for points outside the bounding box
mask_outside = (
    (df["Latitude"] < 41.6) | (df["Latitude"] > 42.1) |
    (df["Longitude"] < 12.1) | (df["Longitude"] > 12.8)
)

# Collect unique Protocollo values for those rows
protocols_latlong_wrong = df.loc[mask_outside, "Protocollo"].unique().tolist()

print(
    f"Number of protocols outside bounding box: {len(protocols_latlong_wrong)}")

Number of protocols outside bounding box: 27


We can delete these latitudes and longitudes as they are definitely incorrect:

In [84]:
# Mask for rows with protocols where the latitude and/or longitude are incorrect:
mask_wrong = df["Protocollo"].isin(protocols_latlong_wrong)

# Count NaNs before
before_lat_na = df["Latitude"].isna().sum()
before_lon_na = df["Longitude"].isna().sum()

# Replace Latitude and Longitude with NaN
df.loc[mask_wrong, ["Latitude", "Longitude"]] = np.nan

# Count NaNs after
after_lat_na = df["Latitude"].isna().sum()
after_lon_na = df["Longitude"].isna().sum()

# Report
print(f"Rows affected: {mask_wrong.sum()}")
print(f"NaN Latitude before: {before_lat_na}, after: {after_lat_na}")
print(f"NaN Longitude before: {before_lon_na}, after: {after_lon_na}")

Rows affected: 29
NaN Latitude before: 2150, after: 2179
NaN Longitude before: 2152, after: 2181


In [85]:
round(df[['Latitude', 'Longitude']].describe(), 5)

,Latitude,Longitude
count,17237.00000,17235.00000
mean,41.88856,12.49200
std,0.04969,0.07401
min,41.63436,12.23918
25%,41.86738,12.45159
50%,41.89451,12.48950
75%,41.91620,12.53600
max,42.09547,12.78931


In [86]:
boxlatlong(df)

The positions with the lowest latitudes are towards Ostia, southwest of the city of Rome, which is currently considered part of the province of Rome. 
The positions with the highest longitudes are towards Zagarolo, to the west of the city of Rome.

Now we can find the median latitude and longitude and use it to request weather data for Rome. 

In [87]:
df['Latitude'].median()

41.89451

In [88]:
df['Longitude'].median()

12.4895

### (O) 'Condizione Atmosferica' Column
How many rows have missing weather data?

In [89]:
def analyse_column(column_name):
    """
    Analyzes value counts, percentages, and unique protocols for a column in the DataFrame.
    Permanently converts empty strings to NaN in the original DataFrame (for string-like columns).

    Args:
        column_name (str): The name of the column to analyze.

    Returns:
        pandas.DataFrame: A DataFrame with 'Count', '% per category', and 
                          'Unique Accidents' for each unique value.
    """
    # Validate input
    if column_name not in df.columns:
        raise ValueError(f"Column '{column_name}' not found in DataFrame")

    s = df[column_name]

    # Count empty strings (via temporary string view; safe for any dtype)
    empty_string_count = (s.astype("string") == "").sum()

    # If the column is object/string/categorical → convert and replace
    if (
        pd.api.types.is_object_dtype(s)
        or pd.api.types.is_string_dtype(s)
        or isinstance(s.dtype, pd.CategoricalDtype)  # modern check
    ):
        df[column_name] = s.astype("string").replace("", pd.NA)
        if empty_string_count > 0:
            print(
                f"Converted {empty_string_count} empty strings to NaN in column '{column_name}'")
    else:
        if empty_string_count > 0:
            print(
                f"Note: counted {empty_string_count} empty strings in non-string column '{column_name}', no replacement performed.")

    cleaned_column = df[column_name]

    # Calculate counts and percentages (include NaNs)
    counts = cleaned_column.value_counts(dropna=False)
    percentages = (cleaned_column.value_counts(
        dropna=False, normalize=True) * 100).round(2)

    # Calculate unique protocols for each category/value
    unique_protocols = []
    for category in counts.index:
        if pd.isna(category):
            mask = cleaned_column.isna()
        else:
            mask = cleaned_column == category
        protocol_count = df.loc[mask, 'Protocollo'].nunique()
        unique_protocols.append(protocol_count)

    # Build result
    result = pd.DataFrame({
        'Count': counts,
        '% per category': percentages,
        'Unique Accidents': unique_protocols
    }, index=counts.index)

    return result

In [90]:
analyse_column('CondizioneAtmosferica')

Converted 1 empty strings to NaN in column 'CondizioneAtmosferica'


,Count,% per category,Unique Accidents
CondizioneAtmosferica,,,
Sereno,14049,72.36,12964
Nuvoloso,3289,16.94,3081
Pioggia in atto,1911,9.84,1758
Nebbia,79,0.41,71
Sole radente,76,0.39,70
Vento forte,8,0.04,6
Grandine in atto,2,0.01,2
Foschia,1,0.01,1
<NA>,1,0.01,1


In [91]:
df['CondizioneAtmosferica'].isna().sum()

np.int64(1)

As we are only missing data for one date and time, we can fill in the information from external weather reports.

#### FINDING INDEPENDENT WEATHER INFORMATION FOR ROME

First, we will find the median latitude and longitude of the accidents, which will then be used in our request for weather data from open-meteo.com. The median is less affected by outliers than the mean. We will request temperature (actual and perceived), humidity, pressure, rainfall, snowfall, cloud cover, wind gusts and wind speed information for the median latitude and longitude of accidents from 01/01/2013 to 31/08/2022. 

As there is now only one accident (row 15187) where the weather conditions ('CondizioneAtmosferica') are missing, we can impute the weather conditions at that accident time using the open-meteo database. 

Weather information request:
- https://archive-api.open-meteo.com/v1/archive?latitude=41.89451&longitude=12.4895&start_date=2013-01-01&end_date=2022-08-31&hourly=temperature_2m,relative_humidity_2m,rain,precipitation,apparent_temperature,snowfall,snow_depth,surface_pressure,weather_code,cloud_cover,wind_speed_10m,wind_gusts_10m&timezone=Europe%2FBerlin
- https://open-meteo.com/en/docs/historical-weather-api  
- latitude  41.89451  (MEDIAN OF ACCIDENT LATITUDE)
- longitude 12.4895 (MEDIAN OF ACCIDENT LONGITUDE)

In [92]:
# Read first few lines to see the structure
with open("C:/Users/lucyq/Dropbox/AMDP/THESIS/open-meteo-weather-data2.csv", 'r') as f:
    for i, line in enumerate(f):
        print(f"Line {i+1}: {line}")
        if i > 5:  # Show first 6 lines
            break

Line 1: latitude,longitude,elevation,utc_offset_seconds,timezone,timezone_abbreviation

Line 2: 41.862915,12.539912,44.0,7200,Europe/Berlin,GMT+2

Line 3: 

Line 4: time,temperature_2m (°C),relative_humidity_2m (%),rain (mm),precipitation (mm),apparent_temperature (°C),snowfall (cm),snow_depth (m),surface_pressure (hPa),weather_code (wmo code),cloud_cover (%),wind_speed_10m (km/h),wind_gusts_10m (km/h)

Line 5: 2013-01-01T00:00,3.0,85,0.00,0.00,0.3,0.00,0.00,1016.6,0,0,4.8,7.6

Line 6: 2013-01-01T01:00,2.3,87,0.00,0.00,-0.4,0.00,0.00,1016.3,0,0,4.1,7.9

Line 7: 2013-01-01T02:00,1.8,88,0.00,0.00,-0.9,0.00,0.00,1016.2,0,0,4.6,9.0



Note:
- LATITUDE requested 41.89451
- LATITUDE supplied 41.862915 

- LONGITUDE requested 12.4895
- LONGITUDE supplied 12.539912

- MEDIAN ACCIDENT POSITION: west of Via dei Serpenti, north of Via Cavour (Monti) - CENTRAL ROME
- WEATHER STATION POSITION: east of Via Appia Nuova, east of Via Tuscolana, south of Via dell'Arco di Travertino (Arco di Travertino) - SOUTH EAST ROME

- Distance between the two positions is 5.5 km (calculated by Google maps using the longitudes and latitudes)

In [93]:
df_weather = pd.read_csv(
    "C:/Users/lucyq/Dropbox/AMDP/THESIS/open-meteo-weather-data2.csv",
    skiprows=3  # Skip the first 3 lines, start reading from line 4
)

In [94]:
df_weather.columns

Index(['time', 'temperature_2m (°C)', 'relative_humidity_2m (%)', 'rain (mm)',
       'precipitation (mm)', 'apparent_temperature (°C)', 'snowfall (cm)',
       'snow_depth (m)', 'surface_pressure (hPa)', 'weather_code (wmo code)',
       'cloud_cover (%)', 'wind_speed_10m (km/h)', 'wind_gusts_10m (km/h)'],
      dtype='object')

In [95]:
df_weather.dtypes

time                          object
temperature_2m (°C)          float64
relative_humidity_2m (%)       int64
rain (mm)                    float64
precipitation (mm)           float64
apparent_temperature (°C)    float64
snowfall (cm)                float64
snow_depth (m)               float64
surface_pressure (hPa)       float64
weather_code (wmo code)        int64
cloud_cover (%)                int64
wind_speed_10m (km/h)        float64
wind_gusts_10m (km/h)        float64
dtype: object

In [96]:
# Convert to datetime and round down to hour
df_weather['time'] = pd.to_datetime(df_weather['time'])

In [97]:
# Localize to Europe/Rome
df_weather['time_local'] = df_weather['time'].dt.tz_localize(
    'Europe/Rome', ambiguous='NaT', nonexistent='shift_forward')

# Convert to UTC
df_weather['time_utc'] = df_weather['time_local'].dt.tz_convert('UTC')

In [98]:
# Round accident time to the nearest hour in UTC
df['DataOraIncidente_utc_rounded'] = (
    df['DataOraIncidente']
    .dt.tz_convert('UTC')
    .dt.round('h')
)

In [99]:
# Left join accident data with weather data based on rounded UTC time of incident
df_merged = pd.merge(
    df,
    df_weather,
    how='left',
    left_on='DataOraIncidente_utc_rounded',
    right_on='time_utc',
    suffixes=('', '_weather')
)

Weather codes: https://www.nodc.noaa.gov/archive/arc0021/0002199/1.1/data/0-data/HTML/WMO-CODE/WMO4677.HTM

Each code gives a short summary of the weather, comparable with the data in the Condizione Atmosferica column. 

In [100]:
df_merged.columns[-16:]

Index(['DataOraIncidente_utc_rounded', 'time', 'temperature_2m (°C)',
       'relative_humidity_2m (%)', 'rain (mm)', 'precipitation (mm)',
       'apparent_temperature (°C)', 'snowfall (cm)', 'snow_depth (m)',
       'surface_pressure (hPa)', 'weather_code (wmo code)', 'cloud_cover (%)',
       'wind_speed_10m (km/h)', 'wind_gusts_10m (km/h)', 'time_local',
       'time_utc'],
      dtype='object')

In [101]:
df_merged[df_merged['CondizioneAtmosferica'].isna()][['Protocollo', 'CondizioneAtmosferica', 'temperature_2m (°C)',
                                                      'relative_humidity_2m (%)', 'rain (mm)', 'precipitation (mm)',
                                                      'apparent_temperature (°C)', 'snowfall (cm)', 'snow_depth (m)',
                                                      'surface_pressure (hPa)', 'weather_code (wmo code)', 'cloud_cover (%)',
                                                      'wind_speed_10m (km/h)', 'wind_gusts_10m (km/h)']].T

,15187
Protocollo,4925779
CondizioneAtmosferica,<NA>
temperature_2m (°C),21.1
relative_humidity_2m (%),62
rain (mm),0.0
precipitation (mm),0.0
apparent_temperature (°C),20.4
snowfall (cm),0.0
snow_depth (m),0.0
surface_pressure (hPa),1007.5


The accident for row 15187 happened while there was no rainfall, a temperature of 21.1 degrees Celsius, 19% cloud cover, wind speed 13 km/h with gusts of 43 km/h. 
The weather code 0 refers to: Cloud development not observed or not observable. 

We should choose between 'sereno' and 'vento forte', given the high wind speeds at the time of the accident.  

We compare the wind speeds and gusts for accidents for which the conditions 'strong wind' (Vento forte) was logged by local police. 

In [102]:
df_merged[df_merged['CondizioneAtmosferica'] == 'Vento forte'][[
    'wind_speed_10m (km/h)', 'wind_gusts_10m (km/h)']].describe()

,wind_speed_10m (km/h),wind_gusts_10m (km/h)
count,8.000000,8.000000
mean,19.437500,41.275000
std,7.004884,14.353471
min,8.300000,15.800000
25%,15.800000,35.300000
50%,18.950000,38.550000
75%,22.900000,51.125000
max,31.500000,59.400000


For comparision, let's take those where the weather was logged by local police as 'serene' (Sereno)

In [103]:
df_merged[df_merged['CondizioneAtmosferica'] == 'Sereno'][[
    'wind_speed_10m (km/h)', 'wind_gusts_10m (km/h)']].describe()

,wind_speed_10m (km/h),wind_gusts_10m (km/h)
count,14049.000000,14049.000000
mean,9.019717,21.204726
std,5.337791,10.489775
min,0.000000,1.800000
25%,5.200000,13.300000
50%,7.800000,18.700000
75%,11.900000,27.400000
max,38.400000,74.900000


In [104]:
# Filter only "Vento forte"
df_vento_forte = df_merged[df_merged['CondizioneAtmosferica'] == 'Vento forte']

# Calculate mean wind speed where CondizioneAtmosferica is NaN
vline_value = df_merged.loc[df_merged['CondizioneAtmosferica'].isna(),
                            'wind_speed_10m (km/h)'].mean()

# Build box plot
fig_windspeed_ventoforte = px.box(
    df_vento_forte,
    x="wind_speed_10m (km/h)",
    width=800,
    height=200,
    title='Wind speed and wind gusts when Condizione Atmosferica = Vento forte'
)

# Add red vertical line
fig_windspeed_ventoforte.add_vline(
    x=vline_value,
    line_color="red",
    line_dash="dash",
    line_width=2,
    annotation_text="NaN CondizioneAtmosferica",
    annotation_position="top left"
)

# Calculate mean wind speed where CondizioneAtmosferica is NaN
vline_value_gusts = df_merged.loc[df_merged['CondizioneAtmosferica'].isna(),
                                  'wind_gusts_10m (km/h)'].mean()

# Build box plot
fig_windgusts_ventoforte = px.box(
    df_vento_forte,
    x='wind_gusts_10m (km/h)',
    width=800,
    height=200
)

# Add red vertical line
fig_windgusts_ventoforte.add_vline(
    x=vline_value_gusts,
    line_color="red",
    line_dash="dash",
    line_width=2,
    annotation_text="NaN CondizioneAtmosferica",
    annotation_position="top left"
)

fig_windspeed_ventoforte.show()
fig_windgusts_ventoforte.show()

Although the wind speed for the accident with missing data is rather low, compared to the other accidents where 'vento forte' was logged, the wind gusts are comparable with those logged for 'vento forte' accidents. 

For comparison, we can look at wind speeds and gusts for 'sereno' accidents:

In [105]:
# Filter only "Vento forte"
df_sereno = df_merged[df_merged['CondizioneAtmosferica'] == 'Sereno']

# Calculate mean wind speed where CondizioneAtmosferica is NaN
vline_value = df_merged.loc[df_merged['CondizioneAtmosferica'].isna(),
                            'wind_speed_10m (km/h)'].mean()

# Build box plot
fig_windspeed_sereno = px.box(
    df_sereno,
    x="wind_speed_10m (km/h)",
    width=800,
    height=200,
    title='Wind speed and wind gusts when Condizione Atmosferica = Sereno'
)

# Add red vertical line
fig_windspeed_sereno.add_vline(
    x=vline_value,
    line_color="red",
    line_dash="dash",
    line_width=2,
    annotation_text="NaN CondizioneAtmosferica",
    annotation_position="top left"
)

# Calculate mean wind speed where CondizioneAtmosferica is NaN
vline_value_gusts = df_merged.loc[df_merged['CondizioneAtmosferica'].isna(),
                                  'wind_gusts_10m (km/h)'].mean()

# Build box plot
fig_windgusts_sereno = px.box(
    df_sereno,
    x='wind_gusts_10m (km/h)',
    width=800,
    height=200
)

# Add red vertical line
fig_windgusts_sereno.add_vline(
    x=vline_value_gusts,
    line_color="red",
    line_dash="dash",
    line_width=2,
    annotation_text="NaN CondizioneAtmosferica",
    annotation_position="top left"
)

fig_windspeed_sereno.show()
fig_windgusts_sereno.show()

Here we can see that when 'sereno' was logged by police, the wind speeds and gust speeds are generally much lower than those observed for the accident with missing data. Therefore, the most appropriate value to insert is 'Vento forte'. 

We insert 'Vento forte' for the accident with missing data:

In [106]:
df_merged['CondizioneAtmosferica'] = df_merged['CondizioneAtmosferica'].fillna(
    "Vento forte")
df_merged['CondizioneAtmosferica'].value_counts(dropna=False)

CondizioneAtmosferica
Sereno              14049
Nuvoloso             3289
Pioggia in atto      1911
Nebbia                 79
Sole radente           76
Vento forte             9
Grandine in atto        2
Foschia                 1
Name: count, dtype: Int64

Now there are no missing values in the 'CondizioneAtmosferica' column.

Police will log the condition most objectively observable (più accertabile). It is also possible that 'CondizioneAtmosferica' is not considered important for the police accident logs, as there seem to be strong winds and strong gusts for a large number of accidents where 'sereno' was logged.

Nevertheless, we will save this feature as a categorical type:

In [107]:
translate_weather = {
    "Sereno": "Clear",
    "Nuvoloso": "Cloudy",
    "Pioggia in atto": "Rain",
    "Nebbia": "Fog",
    "Sole radente": "Blazing sun",
    "Vento forte": "Strong wind",
    "Grandine in atto": "Hail",
    "Foschia": "Haze"
}

# Map
df_merged["weather_condition_noted"] = df_merged["CondizioneAtmosferica"].map(
    translate_weather)

# Build category list from the Italian uniques (preserves encountered order)
italian_cats = list(df_merged["CondizioneAtmosferica"].dropna().unique())
english_cats = [translate_weather[it] for it in italian_cats]

df_merged["weather_condition_noted"] = pd.Categorical(
    df_merged["weather_condition_noted"],
    categories=english_cats,
    ordered=False
)

In [108]:
df_merged['weather_condition_noted'].value_counts(dropna=False)

weather_condition_noted
Clear          14049
Cloudy          3289
Rain            1911
Fog               79
Blazing sun       76
Strong wind        9
Hail               2
Haze               1
Name: count, dtype: int64

In [109]:
df_merged.drop(columns="CondizioneAtmosferica", inplace=True)

### NEW FEATURES - WEATHER

Now our weather data has been merged with the original accident data, we can also create new weather-based features in our df_merged dataframe:

In [110]:
# WET when rain or precipitation greater than zero
df_merged['weather_wet'] = ((df_merged['rain (mm)'] > 0) | (
    df_merged['precipitation (mm)'] > 0)).astype(int)

# SNOWY when snowfall or snow_depth greater than zero
df_merged['weather_snowy'] = ((df_merged['snowfall (cm)'] > 0) | (
    df_merged['snow_depth (m)'] > 0)).astype(int)

# ICY when temperature zero or lower and rain/snow greater than zero
df_merged['weather_icy'] = ((df_merged['temperature_2m (°C)'] <= 0) & (
    df_merged['precipitation (mm)'] > 0)).astype(int)

Although hourly rainfall may have an effect on accidents, other aspects to consider are dry spells before new rain and cumulative rain over the last day. 

In [111]:
# NEW FEATURE: weather_days_since_last_rain_hour
# Identifies dry spells prior to current precipitation

# 1) Obtain a clean, sorted df with non-null time_utc
dfw = (
    df_weather
    .loc[df_weather['time_utc'].notna()]
    .sort_values('time_utc')
    .copy()
)

# 2) Identify rainy hours
rain_mask = dfw['precipitation (mm)'].fillna(0) > 0

# 3) Build rain events table (distinct timestamps when it rained)
if rain_mask.any():
    rain_events = (
        dfw.loc[rain_mask, ['time_utc']]
        .drop_duplicates()
        .rename(columns={'time_utc': 'last_rain_hour'})
        .sort_values('last_rain_hour')
    )

    # 4) Most recent *earlier* rain time for each hour
    dfw = pd.merge_asof(
        dfw,
        rain_events,
        left_on='time_utc',
        right_on='last_rain_hour',
        direction='backward',
        allow_exact_matches=False  # strictly before current hour
    )
else:
    # No rain at all -> no prior rain times
    dfw['last_rain_hour'] = pd.NaT

# 5) Days since last rain
dfw['weather_days_since_last_rain'] = (
    (dfw['time_utc'] - dfw['last_rain_hour']).dt.total_seconds() / (3600 * 24)
)

# 6) Merge back to your accidents join (use the clean dfw with valid time_utc)
df_merged = pd.merge(
    df_merged,
    dfw[['time_utc', 'weather_days_since_last_rain']],
    on='time_utc',
    how='left'
)

Double check to see if the number of days since it last rained are reasonable values:

In [112]:
print(df_merged['weather_days_since_last_rain'].min())
print(df_merged['weather_days_since_last_rain'].max())

0.041666666666666664
35.541666666666664


In [113]:
# NEW FEATURE: weather_cumulative_rain_past_24h
# Returns mm of rain over last 24 hours

# For each hour, calculate sum of rain in the past 24 hours
df_weather['weather_cumulative_rain_past_24h'] = df_weather['rain (mm)'].rolling(
    window=24, min_periods=1).sum()


# Merge back to main DataFrame
df_merged = pd.merge(df_merged, df_weather[[
                     'time_utc', 'weather_cumulative_rain_past_24h']], on='time_utc', how='left')

Double check to see if the values in weather_cumulative_rain_past_24h:

In [114]:
# Find first index where cumulative rain > 0
first_rain_index = df_weather[df_weather['weather_cumulative_rain_past_24h'] > 0].index[0]

# Show next 10 rows to check:
print(df_weather[['time_utc', 'rain (mm)', 'weather_cumulative_rain_past_24h']
                 ].iloc[first_rain_index:first_rain_index+11])

                    time_utc  rain (mm)  weather_cumulative_rain_past_24h
26 2013-01-02 01:00:00+00:00        0.2                               0.2
27 2013-01-02 02:00:00+00:00        0.0                               0.2
28 2013-01-02 03:00:00+00:00        0.1                               0.3
29 2013-01-02 04:00:00+00:00        0.3                               0.6
30 2013-01-02 05:00:00+00:00        0.2                               0.8
31 2013-01-02 06:00:00+00:00        0.0                               0.8
32 2013-01-02 07:00:00+00:00        0.0                               0.8
33 2013-01-02 08:00:00+00:00        0.0                               0.8
34 2013-01-02 09:00:00+00:00        0.0                               0.8
35 2013-01-02 10:00:00+00:00        0.0                               0.8
36 2013-01-02 11:00:00+00:00        0.0                               0.8


### Summary Table - WEATHER FEATURES
Features 
- weather_wet: 1 if rain > 0 or precipitation > 0, else 0

- weather_snowy: 1 if snowfall > 0 or snow_depth > 0, else 0

- weather_icy: 1 if temperature_2m ≤ 0 and precipitation > 0, else 0

- weather_days_since_last_rain	- Days since last rain before accident

- weather_cumulative_rain_past_24h	- Total rain in 24h before accident

The weather data is based on observations from a single weather station at a fixed latitude and longitude in southeast Rome. However, accidents occurred at various locations across the city. Rome’s ring road has a diameter of roughly 23 km, and the ‘Rome’ area stretches southwest as far as Ostia, about 36 km from the ring road’s northeast side. The weather position used is 16km away from the opposite side of the GRA. Therefore we will leave 'CondizioneAtmosferica' as a categorical feature, but also analyse the numerical values provided by the weather centre. 

In [115]:
df_merged.columns

Index(['Protocollo', 'DataOraIncidente', 'Gruppo', 'Localizzazione1',
       'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02', 'Chilometrica',
       'DaSpecificare', 'Latitude', 'Longitude', 'parked_vehicle_involved',
       'driver_injury', 'driver_gender', 'passenger_no_of_females',
       'passenger_no_of_males', 'passenger_no_of_unknown_sex',
       'passengers_killed', 'passengers_uninjured', 'passengers_injured',
       'phase_of_day', 'YEAR', 'MONTH', 'DATE', 'TIME', 'DAY',
       'pedestrian_gender', 'road_conditions', 'road_markings_absent',
       'road_markings_onroad', 'road_markings_signposts',
       'road_markings_temporary', 'road_features',
       'road_markings_traffic_lights', 'road_surface', 'accident_type',
       'TipoStradaDifficulty', 'visibility', 'pedestrian_injury',
       'lighting_insufficient', 'traffic_density', 'vehicle_type',
       'vehicle_unknown', 'hit_and_run', 'DataOraIncidente_utc_rounded',
       'time', 'temperature_2m (°C)', 'relative_humi

In [116]:
df_merged.to_parquet('009_weather_included.parquet', index=False)

In [117]:
df_merged.to_csv('009_weather_included.csv', index=False)

In [ ]:
metadata = {
    'notebook'               : '009 Data cleaning weather.ipynb',
    'step'                   : 'merge hourly weather & engineer weather features',

    # IO lineage
    'initial_parquet_file'   : '008_data_encoding_ordered.parquet',   # expected upstream
    'final_csv_file'         : '009_weather_included.csv',

    # Shapes
    'initial_rows'           : 19_416,
    'final_rows'             : 19_416,
    'final_columns'          : 67,
    'initial_columns_est'    : 45,   # ≈ final_columns - columns_added_count

    # Columns
    'columns_added'          : [
        'DataOraIncidente_utc_rounded','time','time_local','time_utc',
        'temperature_2m (°C)','relative_humidity_2m (%)','rain (mm)','precipitation (mm)',
        'apparent_temperature (°C)','snowfall (cm)','snow_depth (m)','surface_pressure (hPa)',
        'weather_code (wmo code)','cloud_cover (%)','wind_speed_10m (km/h)','wind_gusts_10m (km/h)',
        'weather_condition_noted','weather_wet','weather_snowy','weather_icy',
        'weather_days_since_last_rain','weather_cumulative_rain_past_24h'
    ],
    'columns_removed'        : [],
    'columns_added_count'    : 22,

    # Merge details
    'merge_keys'             : ['DataOraIncidente_utc_rounded' (nearest hour, UTC)],
    'rounding_rule'          : 'accident timestamp → Europe/Rome → UTC → round to nearest hour',
    'merge_type'             : 'left join (1h grid)',
    'merge_completeness'     : {'matched_rows': 19_416, 'match_rate_%': 100.0},
    'utc_rounding_diff_minutes': {'median': 15.0, 'p90': 30.0, 'max': 30.0},

    # Weather features (units inferred from column names)
    'weather_vars_units'     : {
        'temperature_2m (°C)': '°C',
        'apparent_temperature (°C)': '°C',
        'relative_humidity_2m (%)': '%',
        'precipitation (mm)': 'mm',
        'rain (mm)': 'mm',
        'cloud_cover (%)': '%',
        'wind_speed_10m (km/h)': 'km/h',
        'wind_gusts_10m (km/h)': 'km/h',
        'surface_pressure (hPa)': 'hPa',
        'snowfall (cm)': 'cm',
        'snow_depth (m)': 'm',
        'weather_code (wmo code)': 'WMO code'
    },

    # Engineered indicators & lags
    'engineered_features'    : [
        'weather_wet','weather_snowy','weather_icy',
        'weather_days_since_last_rain','weather_cumulative_rain_past_24h'
    ],

    # QA snapshots
    'qa_missing'             : {
        'DataOraIncidente_utc_rounded': 0,
        'weather_code (wmo code)': 0,
        'temperature_2m (°C)': 0,
        'precipitation (mm)': 0,
        'weather_days_since_last_rain': 5
    },
    'qa_weather_code_unique' : 12,
    'qa_weather_condition_noted_top' : {
        'Clear': 14049, 'Cloudy': 3289, 'Rain': 1911, 'Fog': 79,
        'Blazing sun': 76, 'Strong wind': 9, 'Hail': 2, 'Haze': 1
    },
    'qa_rates'               : {
        'wet_rate'  : 0.1693,
        'snowy_rate': 0.000824,
        'icy_rate'  : 0.0
    },
    'qa_ranges'              : {
        'temperature_2m (°C)': {'min': -8.7, 'median': 15.8, 'max': 39.9},
        'apparent_temperature (°C)': {'min': -13.4, 'median': 14.8, 'max': 40.4},
        'relative_humidity_2m (%)': {'min': 12.0, 'median': 71.0, 'max': 100.0},
        'precipitation (mm)': {'min': 0.0, 'median': 0.0, 'max': 15.1},
        'wind_speed_10m (km/h)': {'min': 0.0, 'median': 8.3, 'max': 45.4},
        'wind_gusts_10m (km/h)': {'min': 1.8, 'median': 19.8, 'max': 92.5},
        'cloud_cover (%)': {'min': 0.0, 'median': 35.0, 'max': 100.0},
        'surface_pressure (hPa)': {'min': 977.1, 'median': 1010.2, 'max': 1032.4},
        'snowfall (cm)': {'min': 0.0, 'median': 0.0, 'max': 0.63},
        'snow_depth (m)': {'min': 0.0, 'median': 0.0, 'max': 0.07}
    },

    # Time coverage
    'accident_time_range'    : {'min_local': '2013-01-01T02:15:+01:00', 'max_local': '2022-08-31T23:00:+02:00'},
    'weather_time_range_utc' : {'min': '2013-01-01T01:00:00Z', 'max': '2022-08-31T21:00:00Z'},

    # Decisions
    'decisions_made'         : [
        'Rounded accident timestamps to the nearest UTC hour to align with hourly weather.',
        'Performed left join to keep all accidents; verified 100% weather match.',
        'Created wet/snowy/icy flags and rainfall lag features (days since last rain, 24h cumulative).',
        'Kept weather variables with units in column names for clarity.'
    ]
}
